<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/HW/HW5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 5: Classifying Stars with Machine Learning (42 pts)

## Learning Outcomes

- Understand how **Machine Learning** works: learning patterns from data instead of writing explicit rules
- See how **physical properties of stars** (temperature, color, luminosity) map to classification categories
- Use **scikit-learn** to train and compare classifiers (Decision Tree, k-Nearest Neighbors, Random Forest)
- Evaluate classifiers using **accuracy, confusion matrices, and feature importance**
- Understand when and why **feature scaling** matters for distance-based algorithms
- Create **publication-quality plots** of model comparisons and evaluation metrics
- Reflect on the difference between **Machine Learning and Artificial Intelligence**
- Submit homework via **fork and pull request**

## Development Environment

You have two options for working on this homework:

### Option A: GitHub Codespaces (recommended)

**GitHub Codespaces** gives you a cloud-based VS Code environment with Python, Jupyter, and all dependencies pre-installed — no setup needed.

> **If you have an existing Codespace from HW4**: you need to create a **new** one for HW5. The environment now includes `scikit-learn` which wasn't in the HW4 setup. See [Issue #69](https://github.com/ubsuny/PHY386/issues/69) for step-by-step instructions on saving your work before deleting the old Codespace.
>
> **Quick alternative**: run `pip install -r requirements/requirements_HW2026.txt` in your existing Codespace terminal.

1. **Fork the repository**: Go to [github.com/ubsuny/PHY386](https://github.com/ubsuny/PHY386) → click **"Fork"**
2. **Sync your fork**: On your fork's page, click **"Sync fork"** → **"Update branch"**
3. **Open a Codespace**: Green **"Code"** button → **"Codespaces"** tab → **"Create codespace on Homework2026"**
4. **Create your branch**:
   - **In VS Code**: Click the branch name in the bottom-left corner → **Create new branch** → type `yourname-hw5`
   - **Or in the terminal**: `git checkout -b yourname-hw5`
5. Work on `2026/HW/yourname/HW5.ipynb` using the built-in Jupyter extension

### Option B: Google Colab

Click the **"Open in Colab"** badge at the top of this notebook. When done, use **File → Save a copy to GitHub** to push to your fork.

---

**Either way, you must submit via the fork workflow** (see Submission section at the end).

## Background: From Starlight to Classification

### Why Stars Have Different Colors

If you look up at the night sky, stars aren't all the same. Some glow red, others appear white or bluish. This isn't random — a star's color is determined by its **surface temperature**, following the same blackbody radiation physics you've seen in modern physics.

A cool star (~3,000 K) emits most of its light in the red part of the spectrum, while a hot star (~30,000 K) peaks in the blue/ultraviolet. This is Wien's displacement law in action:

$$\lambda_{\text{max}} = \frac{b}{T} \approx \frac{2.898 \times 10^{-3}\,\text{m}\cdot\text{K}}{T}$$

Astronomers quantify this color using the **B-V color index** — the difference in brightness measured through a Blue filter versus a Visual (green-yellow) filter. A negative B-V means the star is bluer (hotter); a positive B-V means it's redder (cooler):

| B-V | Color | Temperature | Example |
|-----|-------|------------|----------|
| −0.3 | Blue | ~30,000 K | Spica |
| 0.0 | Blue-white | ~10,000 K | Vega |
| +0.4 | White | ~7,500 K | Procyon |
| +0.6 | Yellow-white | ~6,000 K | The Sun |
| +0.8 | Orange | ~5,000 K | Arcturus |
| +1.5 | Red | ~3,500 K | Betelgeuse |

### The Hertzsprung-Russell Diagram

The most important diagnostic tool in stellar astrophysics is the **Hertzsprung-Russell (HR) diagram**, which plots luminosity (or absolute magnitude) against temperature (or color). When you plot thousands of stars, they don't fill the diagram uniformly — they cluster into distinct groups:

![Stellar Life Cycles](https://raw.githubusercontent.com/ubsuny/PHY386/Homework2026/media/Star_life_cycles_red_dwarf_en.svg)

*Stellar evolution of low-mass (left) and high-mass (right) stars. By cmglee, NASA GSFC, CC BY-SA 4.0.*

- **Main Sequence**: The diagonal band from hot/bright (upper left) to cool/dim (lower right). This is where stars spend most of their lives, fusing hydrogen in their cores. The Sun is a main sequence star.
- **Red Giants/Supergiants**: Stars that have exhausted their core hydrogen and expanded enormously. They're cool (red) but very luminous because of their huge surface area.
- **White Dwarfs**: The remnant cores of dead low-mass stars. They're hot but extremely dim because they're tiny (Earth-sized).

The fact that stars cluster into these groups — rather than being randomly scattered — is what makes **classification** possible. If we measure a star's color and brightness, we should be able to determine what type of star it is.

### What is Machine Learning?

Imagine you're an astronomer who has measured the temperature, luminosity, radius, and magnitude of thousands of stars, and you know the type of each one (main sequence, red giant, white dwarf, etc.). You could try to write classification rules by hand:

```python
if temperature > 8000 and luminosity < 0.1:
    star_type = 'White Dwarf'
elif luminosity > 10000:
    star_type = 'Supergiant'
elif ...
```

But this gets complicated fast. Where exactly do you draw the boundaries? What if two types overlap in temperature but differ in luminosity? What if you have 20 features instead of 2?

**Machine Learning** takes a fundamentally different approach: instead of writing the rules, you give the algorithm **examples** (labeled data) and let it **discover the rules** automatically.

### The Supervised Learning Workflow

In **supervised learning**, each data point has:
- **Features** ($X$): the measurements (e.g., temperature, luminosity, color index)
- **Label** ($y$): the correct answer (e.g., star type)

The workflow has four steps:

1. **Split** your data into a **training set** (what the model learns from) and a **test set** (what you evaluate on). This is crucial — if you test on the same data you trained on, you don't know if the model actually learned generalizable patterns or just memorized the training data.

2. **Train** the model on the training set: the algorithm examines many (feature, label) pairs and discovers patterns.

3. **Predict** on the test set: the model sees new features it hasn't trained on and predicts labels.

4. **Evaluate**: compare predictions to the true labels. How often was the model correct?

### Three Classification Algorithms

In this homework, you'll use three different algorithms. Each approaches the classification problem differently:

**Decision Tree** — Asks a series of yes/no questions about the features to split the data. For example: "Is temperature > 8000 K? If yes, check luminosity. If luminosity < 0.1, predict White Dwarf." The algorithm automatically finds the best questions and thresholds. Easy to interpret, but can memorize noise in the training data (called **overfitting**).

**k-Nearest Neighbors (KNN)** — For a new data point, finds the $k$ closest points in the training set (by distance in feature space) and takes a majority vote. If most of a star's 5 nearest neighbors are red giants, predict red giant. Simple and intuitive, but **sensitive to feature scales** — if temperature ranges from 2,000 to 40,000 but luminosity from 0.0001 to 800,000, the temperature differences get drowned out.

**Random Forest** — Trains many decision trees, each on a random subset of data and features, then takes a majority vote. This is an **ensemble method** — combining many weak models into a strong one. Less likely to overfit than a single decision tree. Also tells you which features matter most (feature importance).

### The scikit-learn API

scikit-learn provides a consistent interface for all classifiers. Every algorithm follows the same `fit` → `predict` → `score` pattern:

```python
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# 1. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# 2. Train
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

# 3. Predict and evaluate
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)  # fraction of correct predictions
```

This uniformity means you can swap in any classifier — `KNeighborsClassifier()`, `RandomForestClassifier()` — and the code is nearly identical. Only the algorithm changes; the interface stays the same.

### Evaluating a Classifier: The Confusion Matrix

Accuracy (fraction of correct predictions) is a useful single number, but it doesn't tell you *which* classes the model struggles with. A **confusion matrix** shows this:

- Each **row** is a true class
- Each **column** is a predicted class  
- **Diagonal entries** = correct predictions
- **Off-diagonal entries** = mistakes

For example, if a model confuses Red Giants with Supergiants, you'll see nonzero entries where the Red Giant row meets the Supergiant column. This makes physical sense — both are cool, luminous stars.

---

## Worked Example: Classifying Star Types

Let's walk through the complete ML pipeline using a dataset of 240 stars with 6 known types. This is the same star type dataset from the background section — we'll use it here to demonstrate every step, so you can see exactly how the pieces fit together before applying them to a different dataset in the homework tasks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# --- Publication-quality plot settings ---
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'figure.figsize': (6, 4),
})

### Step 1: Load and Explore the Data

First, load the dataset and understand what we're working with.

In [ ]:
# Load the star type dataset
try:
    df_stars = pd.read_csv('data/stars.csv')
except FileNotFoundError:
    df_stars = pd.read_csv('https://raw.githubusercontent.com/ubsuny/PHY386/Homework2026/data/stars.csv')

# Map the integer types to meaningful names
type_names = {
    0: 'Red Dwarf',
    1: 'Brown Dwarf',
    2: 'White Dwarf',
    3: 'Main Sequence',
    4: 'Supergiant',
    5: 'Hypergiant'
}
df_stars['TypeName'] = df_stars['Type'].map(type_names)

print(f"Dataset: {df_stars.shape[0]} stars, {df_stars.shape[1]} columns")
print(f"\nFeatures: {list(df_stars.columns)}")
print(f"\nStar types:\n{df_stars['TypeName'].value_counts().sort_index()}")
print(f"\nSummary statistics:")
df_stars[['Temperature', 'L', 'R', 'A_M']].describe()

Notice the huge range of values: Temperature goes from ~1,939 to ~40,000 K, while Luminosity ranges from 0.00008 to 850,000 $L_\odot$. This range of scales will matter later.

### Step 2: Visualize — Build an HR Diagram

Before training any model, we should *look* at the data. Plotting Temperature vs. Luminosity — an HR diagram — shows us whether the star types are visually separable.

In [ ]:
# Create an HR diagram colored by star type
fig, ax = plt.subplots(figsize=(6, 5))

# Use distinct colors for each type
colors = ['#d62728', '#8c564b', '#9467bd', '#2ca02c', '#ff7f0e', '#1f77b4']
for star_type in sorted(df_stars['Type'].unique()):
    mask = df_stars['Type'] == star_type
    ax.scatter(
        df_stars.loc[mask, 'Temperature'],
        df_stars.loc[mask, 'L'],
        c=colors[star_type],
        label=type_names[star_type],
        alpha=0.7,
        edgecolors='k',
        linewidths=0.3,
        s=30
    )

ax.set_xscale('log')
ax.set_yscale('log')
ax.invert_xaxis()  # HR diagram convention: hot stars on the left
ax.set_xlabel('Temperature (K)')
ax.set_ylabel(r'Luminosity ($L/L_\odot$)')
ax.set_title('HR Diagram — Star Type Dataset')
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.show()

The star types occupy distinct regions of the HR diagram — exactly as expected from stellar physics. This is promising for classification: if the types are well-separated in feature space, a classifier should be able to learn the boundaries.

### Step 3: Prepare Features and Split the Data

In [ ]:
def prepare_star_features(
    df: pd.DataFrame,
    test_size: float = 0.3,
    random_state: int = 42
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Extract features and labels from the star dataset, then split.

    Uses Temperature, Luminosity (L), Radius (R), and Absolute Magnitude (A_M)
    as features, and Type as the label.

    Parameters
    ----------
    df : pd.DataFrame
        Star dataset with columns Temperature, L, R, A_M, Type.
    test_size : float
        Fraction of data for the test set.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        X_train, X_test, y_train, y_test.
    """
    feature_cols = ['Temperature', 'L', 'R', 'A_M']
    X = df[feature_cols].values
    y = df['Type'].values
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


X_train, X_test, y_train, y_test = prepare_star_features(df_stars)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Features:     {X_train.shape[1]} (Temperature, L, R, A_M)")

### Step 4: Train a Decision Tree and Evaluate

In [ ]:
# Train
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Evaluate
print(f"Training accuracy: {dt.score(X_train, y_train):.2f}")
print(f"Test accuracy:     {dt.score(X_test, y_test):.2f}")

# Detailed classification report
y_pred_dt = dt.predict(X_test)
print(f"\n{classification_report(y_test, y_pred_dt, target_names=list(type_names.values()))}")

Training accuracy of 1.00 means the tree perfectly memorized the training data — this is typical for an unconstrained decision tree. What matters is the **test accuracy**: how well it generalizes to data it hasn't seen.

### Step 5: Plot the Confusion Matrix

In [ ]:
def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list[str],
    title: str = 'Confusion Matrix'
) -> None:
    """Plot a confusion matrix as a heatmap.

    Parameters
    ----------
    y_true : np.ndarray
        True labels.
    y_pred : np.ndarray
        Predicted labels.
    class_names : list[str]
        Names for each class.
    title : str
        Plot title.
    """
    cm = confusion_matrix(y_true, y_pred)
    n = len(class_names)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='viridis')
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)

    # Numeric annotations
    for i in range(n):
        for j in range(n):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color)

    plt.colorbar(im, ax=ax, label='Count')
    plt.tight_layout()
    plt.show()


plot_confusion_matrix(y_test, y_pred_dt, list(type_names.values()),
                      title='Decision Tree — Star Types')

If most of the counts are on the diagonal, the model is doing well. Any off-diagonal entries tell you which types get confused — and you can ask: does this confusion make physical sense?

### Step 6: Compare Multiple Classifiers

Let's train KNN and Random Forest on the same data and compare.

In [ ]:
# KNN — note the lower accuracy on unscaled data!
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print(f"KNN test accuracy (unscaled): {knn.score(X_test, y_test):.2f}")

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f"Random Forest test accuracy:  {rf.score(X_test, y_test):.2f}")
print(f"Decision Tree test accuracy:  {dt.score(X_test, y_test):.2f}")

KNN performs much worse — why? Because it uses **distance** to find neighbors, and our features are on wildly different scales. Temperature ranges up to 40,000 while Luminosity can be 0.0001. The distance calculation is dominated by whichever feature has the largest numbers.

**Feature scaling** (standardizing each feature to mean=0, std=1) fixes this:

In [ ]:
# Scale features for KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on training data only!
X_test_scaled = scaler.transform(X_test)          # apply same transform to test data

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
print(f"KNN test accuracy (scaled):   {knn_scaled.score(X_test_scaled, y_test):.2f}")
print(f"\nLesson: always scale features for distance-based algorithms like KNN!")

### Step 6b: Visualizing Decision Boundaries

Accuracy numbers tell you *how well* a classifier works, but not *how* it works. A **decision boundary plot** shows exactly how each algorithm partitions the feature space — which regions get assigned to which star type.

To visualize this in 2D, we pick just two features (Temperature and Absolute Magnitude), create a fine grid over the feature plane, and color each grid point by the classifier's prediction. The training data points are overlaid so you can see how the boundaries relate to the actual stars.

Notice the differences:
- **Decision Tree**: axis-aligned rectangular cuts (it can only ask "is feature > threshold?")
- **KNN**: irregular, locally-adaptive boundaries that follow the shape of the data
- **Random Forest**: smoother boundaries than a single tree, because it averages many trees

In [ ]:
# Decision boundary visualization using only 2 features (Temperature, Abs. Magnitude)
# so we can plot the classifier's decisions as colored regions in 2D.

# Use only Temperature (col 0) and Abs. Magnitude (col 3) for visualization
X_2d_train = X_train[:, [0, 3]]  # Temperature, A_M
X_2d_test = X_test[:, [0, 3]]

# Scale for KNN (and use same scale for consistent grid)
scaler_2d = StandardScaler()
X_2d_train_sc = scaler_2d.fit_transform(X_2d_train)
X_2d_test_sc = scaler_2d.transform(X_2d_test)

# Train all three classifiers on the 2-feature data
classifiers = {
    'Decision Tree': DecisionTreeClassifier(random_state=42).fit(X_2d_train_sc, y_train),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5).fit(X_2d_train_sc, y_train),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42).fit(X_2d_train_sc, y_train),
}

# Create a mesh grid over the scaled feature space
x_min, x_max = X_2d_train_sc[:, 0].min() - 0.5, X_2d_train_sc[:, 0].max() + 0.5
y_min, y_max = X_2d_train_sc[:, 1].min() - 0.5, X_2d_train_sc[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min, y_max, 200))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
colors_scatter = ['#d62728', '#8c564b', '#9467bd', '#2ca02c', '#ff7f0e', '#1f77b4']

for ax, (name, clf) in zip(axes, classifiers.items()):
    # Predict on every grid point → colored background
    Z = clf.predict(grid_points).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, levels=np.arange(-0.5, 6, 1),
                colors=colors_scatter, antialiased=True)

    # Overlay training data
    for star_type in sorted(np.unique(y_train)):
        mask = y_train == star_type
        ax.scatter(X_2d_train_sc[mask, 0], X_2d_train_sc[mask, 1],
                   c=colors_scatter[star_type], label=type_names[star_type],
                   edgecolors='k', linewidths=0.3, s=25, alpha=0.8)

    ax.set_title(f'{name}\n(acc={clf.score(X_2d_test_sc, y_test):.2f})')
    ax.set_xlabel('Temperature (scaled)')
    if ax == axes[0]:
        ax.set_ylabel('Abs. Magnitude (scaled)')

axes[0].legend(fontsize=7, loc='upper left', ncol=2)
fig.suptitle('Decision Boundaries — How Each Classifier Partitions Feature Space',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("Notice: Decision Tree makes rectangular cuts (axis-aligned splits).")
print("KNN creates irregular boundaries that closely follow local data density.")
print("Random Forest smooths the boundaries by averaging many trees.")

### Step 7: Feature Importance from Random Forest

Which features does the model rely on most? Random Forest can tell us:

In [ ]:
feature_names = ['Temperature', 'Luminosity', 'Radius', 'Abs. Magnitude']
importances = rf.feature_importances_

fig, ax = plt.subplots()
y_pos = range(len(feature_names))
ax.barh(y_pos, importances, color='steelblue')
ax.set_yticks(y_pos)
ax.set_yticklabels(feature_names)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest — Which Features Matter?')
plt.tight_layout()
plt.show()

print("Radius and Absolute Magnitude are the most important features.")
print("This makes physical sense: a star's radius determines whether it's a")
print("dwarf or giant, and absolute magnitude encodes intrinsic brightness.")

---

## Part 0: ML vs AI — Before You Start (4 pts)

**Before doing any of the tasks below**, ask an LLM (ChatGPT, Claude, Gemini, etc.) the following question:

> *"What is the difference between Machine Learning and Artificial Intelligence? Explain it in the context of classifying astronomical objects."*

**Task 0.1 (4 pts):** Paste the LLM's full response in the markdown cell below. Note which LLM you used and the date.

*Your LLM response here. Include the LLM name and date.*

---

## Part 1: Data Exploration (8 pts)

For the homework tasks, you will use a **different dataset**: the **Hipparcos Giants & Dwarfs** catalog. This contains 500 stars from the [Hipparcos satellite](https://en.wikipedia.org/wiki/Hipparcos) with a binary classification: is the star a **giant** (evolved, luminous star) or a **dwarf** (main-sequence or sub-main-sequence star)?

The features are:

| Column | Description | Unit |
|--------|-------------|------|
| `Vmag` | Apparent visual magnitude | mag |
| `Plx` | Parallax (inversely related to distance) | mas |
| `e_Plx` | Parallax error | mas |
| `B-V` | Color index (blue minus visual magnitude) | mag |
| `SpType` | Spectral type string (e.g., "K5III", "F9V") | — |
| `Amag` | Absolute magnitude (intrinsic brightness) | mag |
| `TargetClass` | 0 = Giant, 1 = Dwarf | — |

Note: In the spectral type string, Roman numerals indicate luminosity class: **V** = dwarf/main sequence, **III** = giant, **I** = supergiant.

**Task 1.1 (3 pts):** Load `data/hipparcos_giants_dwarfs.csv` with pandas. Print `df.shape`, `df.describe()`, and `df.head(10)`. Write a function `load_and_summarize` that does this and returns the DataFrame. Print the class distribution using `value_counts()`.

In [ ]:
# Task 1.1: Your code here
# Local/Codespaces: pd.read_csv('data/hipparcos_giants_dwarfs.csv')
# Colab fallback:   pd.read_csv('https://raw.githubusercontent.com/ubsuny/PHY386/Homework2026/data/hipparcos_giants_dwarfs.csv')


**Task 1.2 (5 pts):** Create a **2-panel figure** (`figsize=(10, 4)`) showing:

- **Left panel**: B-V color index vs. Absolute Magnitude (`Amag`), colored by `TargetClass` (0=Giant, 1=Dwarf). This is a **color-magnitude diagram** — the observational equivalent of the HR diagram.
- **Right panel**: Parallax (`Plx`) vs. Apparent Magnitude (`Vmag`), colored by `TargetClass`.

Requirements:
- **Invert the y-axis** on the left panel (brighter stars = lower magnitude numbers, by convention)
- Use two distinct colors with a legend showing "Giant" and "Dwarf"
- Label all axes with units

In [ ]:
# Task 1.2: Your code here


---

## Part 2: Train a Decision Tree Classifier (10 pts)

**Task 2.1 (4 pts):** Write a function that prepares the features and splits the data. Use the 4 numerical columns (`Vmag`, `Plx`, `B-V`, `Amag`) as features and `TargetClass` as the label.

```python
def prepare_features(
    df: pd.DataFrame,
    test_size: float = 0.3,
    random_state: int = 42
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Extract features and labels, then split into train/test sets.

    Uses Vmag, Plx, B-V, and Amag as features and TargetClass as the label.

    Parameters
    ----------
    df : pd.DataFrame
        Hipparcos dataset.
    test_size : float
        Fraction of data for the test set.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        X_train, X_test, y_train, y_test arrays.
    """
    ...
```

In [ ]:
# Task 2.1: Your code here


**Task 2.2 (3 pts):** Train a `DecisionTreeClassifier(random_state=42)` on the training set. Print both the **training accuracy** and **test accuracy**. Is there a gap? What might it mean?

In [ ]:
# Task 2.2: Your code here


**Task 2.3 (3 pts):** Compute and plot the **confusion matrix** as a heatmap (follow the worked example pattern). Use `viridis` colormap. Label axes as "Giant" and "Dwarf". Add numeric annotations in each cell.

In [ ]:
# Task 2.3: Your code here


---

## Part 3: Compare Classifiers (12 pts)

**Task 3.1 (4 pts):** Train a `KNeighborsClassifier(n_neighbors=5)` on the same train/test split. Print training and test accuracy.

Then try again with `StandardScaler` applied to the features (as shown in the worked example). Print both accuracies. Is there a difference? Why?

*Remember*: fit the scaler on the training set only, then transform both train and test.

In [ ]:
# Task 3.1: Your code here


**Task 3.2 (4 pts):** Train a `RandomForestClassifier(n_estimators=100, random_state=42)`. Print training and test accuracy.

In [ ]:
# Task 3.2: Your code here


**Task 3.3 (4 pts):** Write a function `compare_classifiers` that trains all three classifiers (use scaled features for KNN) and returns a dictionary of test accuracies. Create a **bar chart** comparing them.

```python
def compare_classifiers(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray
) -> dict[str, float]:
    """Train Decision Tree, KNN (scaled), and Random Forest; return test accuracies.

    Parameters
    ----------
    X_train : np.ndarray
        Training features.
    X_test : np.ndarray
        Test features.
    y_train : np.ndarray
        Training labels.
    y_test : np.ndarray
        Test labels.

    Returns
    -------
    dict[str, float]
        Dictionary mapping classifier name to test accuracy.
    """
    ...
```

In a markdown cell below the plot, write 2-3 sentences explaining which classifier performed best and why you think that is.

In [ ]:
# Task 3.3: Your code here


*Your discussion here (2-3 sentences): Which classifier performed best? Why?*

---

## Part 4: Feature Importance (4 pts)

**Task 4.1 (4 pts):** Extract `feature_importances_` from the Random Forest model. Create a **horizontal bar chart** showing which features are most important for the giant/dwarf classification. Label bars with the feature names (Vmag, Parallax, B-V Color, Abs. Magnitude).

In a markdown cell, discuss (2-3 sentences): Does the most important feature make physical sense? Why would that feature be most useful for distinguishing giants from dwarfs? (*Hint*: think about what the B-V color index and absolute magnitude tell you about a star's position on the HR diagram.)

In [ ]:
# Task 4.1: Your code here


*Your discussion here (2-3 sentences): Does the most important feature make physical sense?*

---

## Part 5: ML vs AI — After Completing the Tasks (4 pts)

Now that you've trained classifiers, evaluated them, and analyzed feature importance, revisit the LLM's answer from Part 0.

**Task 5.1 (4 pts):** In the markdown cell below, write a reflection (4-6 sentences) addressing:

1. Based on what you just did — was this "Machine Learning" or "Artificial Intelligence"? Or both? Explain.
2. Look back at the LLM's original explanation from Part 0. Was its distinction between ML and AI accurate? Did it overstate or understate the capabilities of what you implemented?
3. What surprised you about the ML process compared to what you expected before starting?

*Your reflection here (4-6 sentences).*

---

## Submission: Fork Workflow & PR

### Commit Your Work

**Using VS Code (recommended):**
1. Open the **Source Control** panel (branch icon in the left sidebar, or <kbd>Ctrl</kbd>+<kbd>Shift</kbd>+<kbd>G</kbd>)
2. Review your changes — files appear under "Changes"
3. Click the **+** next to each file to stage it (or **+** next to "Changes" to stage all)
4. Type a descriptive commit message in the text box
5. Click the **✓ Commit** button
6. Click **Sync Changes** (or the cloud upload icon) to push to your fork

**Using the terminal:**
```bash
git add 2026/HW/yourname/HW5.ipynb
git commit -m "Train Decision Tree on Hipparcos giant/dwarf dataset"
git push origin yourname-hw5
```

**Good commit messages:**
- `Train Decision Tree on Hipparcos giant/dwarf dataset`
- `Add confusion matrix and classifier comparison bar chart`
- `Add ML vs AI reflection with LLM evaluation`

**Bad commit messages:** `update`, `fix`, `stuff`, `done`

### Open a Pull Request

1. Go to `github.com/yourname/PHY386`
2. Click **"Compare & pull request"**
3. **Important**: Set base repository to `ubsuny/PHY386` and base branch to `Homework2026`
4. Title: `HW5 - 2026 - yourusername`
5. Fill out the PR template:
   - Add label: `homework-in-progress` or `homework-final`
   - Assign reviewer: `@laserlab`
   - Set milestone: `HW5-2026`

### Checklist

- [ ] Forked the repository and synced with upstream
- [ ] Created branch `yourname-hw5` from `Homework2026`
- [ ] Completed Part 0 (LLM response on ML vs AI)
- [ ] Completed Part 1 (data exploration + color-magnitude diagram)
- [ ] Completed Part 2 (Decision Tree + confusion matrix)
- [ ] Completed Part 3 (KNN with scaling + RF + bar chart + discussion)
- [ ] Completed Part 4 (feature importance + discussion)
- [ ] Completed Part 5 (ML vs AI reflection)
- [ ] All functions have **type annotations** and **NumPy-style docstrings**
- [ ] All plots have labeled axes with units
- [ ] All commit messages are descriptive
- [ ] PR opened with correct title, label, reviewer, and milestone

## Resources

- [scikit-learn User Guide — Classification](https://scikit-learn.org/stable/supervised_learning.html)
- [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
- [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [Hertzsprung-Russell Diagram — Wikipedia](https://en.wikipedia.org/wiki/Hertzsprung%E2%80%93Russell_diagram)
- [Stellar Classification — Wikipedia](https://en.wikipedia.org/wiki/Stellar_classification)
- [Hipparcos Catalogue — Wikipedia](https://en.wikipedia.org/wiki/Hipparcos_catalogue)
- [GitHub Codespaces Documentation](https://docs.github.com/en/codespaces)
- [GitHub Forking Workflow](https://docs.github.com/en/pull-requests/collaborating-with-pull-requests/working-with-forks)